In [7]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import requests
import numpy as np
from src.pipeline.clean_datas import clean_datas_history

#from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    roc_auc_score,
    roc_curve,
    auc,
    ConfusionMatrixDisplay
)

In [2]:
df_train = pd.read_csv("../datas/fraudTrain.csv")

df_clean_train = clean_datas_history(df_train)
df_clean_train.sample(10)

,category,amt,gender,state,city_pop,is_fraud,trans_hour,trans_day,trans_month,age,distance_km,customer_job_category
979399,grocery_pos,111.03,F,IL,324,0,3,2,2,50,103,science
29892,health_fitness,26.74,M,TX,20328,0,18,18,1,41,80,tech
1242840,entertainment,89.25,F,VA,976,0,23,1,6,27,78,tech
916168,grocery_pos,105.24,M,NY,1453,0,1,30,12,52,117,health
472397,shopping_net,82.14,M,TX,2395,0,9,27,7,33,30,tech
292563,misc_net,129.89,F,TX,144160,0,5,24,5,42,58,management
1086360,shopping_pos,9.93,M,OH,2644,0,18,27,3,45,98,transport
53371,kids_pets,9.41,F,OH,2644,0,15,1,2,43,110,tech
1087331,grocery_pos,86.33,M,CO,277,0,7,28,3,41,76,management
944648,food_dining,51.86,F,OH,269,0,17,12,1,61,39,tech


In [3]:
df_test = pd.read_csv("../datas/fraudTest.csv")

df_clean_test = clean_datas_history(df_test)
df_clean_test.sample(10)

,category,amt,gender,state,city_pop,is_fraud,trans_hour,trans_day,trans_month,age,distance_km,customer_job_category
304633,gas_transport,55.46,F,NM,247,0,0,14,10,65,57,tech
237490,kids_pets,66.08,M,MI,673342,0,22,14,9,59,75,management
190632,gas_transport,49.30,M,TX,1595797,0,0,27,8,31,91,construction
437567,grocery_pos,104.30,F,WY,100,0,8,6,12,52,76,education
398537,health_fitness,123.16,F,GA,1293,0,13,24,11,63,16,tech
359768,shopping_pos,6.04,M,NC,69793,0,4,8,11,73,4,tech
426551,shopping_pos,149.57,M,PA,168,0,11,3,12,48,86,management
541698,food_dining,5.57,F,PA,166081,0,19,28,12,28,26,science
442184,grocery_pos,66.49,M,VA,43102,0,3,7,12,76,104,tech
355986,grocery_pos,95.64,F,UT,46,0,5,6,11,39,30,construction


In [4]:
feature_target = "is_fraud"

X_test = df_clean_test.drop(columns=[feature_target])
y_test = df_clean_test[feature_target]

X_train = df_clean_train.drop(columns=[feature_target])
y_train = df_clean_train[feature_target]

In [5]:
numeric_features = []
categorical_features = []

for i,t in X_train.dtypes.items():
    if ('float' in str(t)) or ('int' in str(t)) :
        numeric_features.append(i)
    else :
        categorical_features.append(i)
        
print('Found numeric features ', numeric_features)
print('Found categorical features ', categorical_features)

Found numeric features  ['amt', 'city_pop', 'trans_hour', 'trans_day', 'trans_month', 'age', 'distance_km']
Found categorical features  ['category', 'gender', 'state', 'customer_job_category']


In [6]:
numeric_transformer = Pipeline(
    steps=[
        ('scaler', StandardScaler())
    ])

categorical_transformer = Pipeline(
    steps=[
        ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
    ])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [8]:
# Calcul du ratio pour compenser le déséquilibre
scale = (y_train == 0).sum() / (y_train == 1).sum()

pipeline_xgb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(eval_metric="logloss", scale_pos_weight=scale))
])

pipeline_xgb.fit(X_train, y_train)

# Prédiction
y_pred_xgb = pipeline_xgb.predict(X_test)

# Évaluation
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       1.00      0.99      1.00    553574
           1       0.36      0.92      0.52      2145

    accuracy                           0.99    555719
   macro avg       0.68      0.96      0.76    555719
weighted avg       1.00      0.99      0.99    555719



In [11]:
param_grid = {
    'classifier__n_estimators': [100, 300],
    'classifier__max_depth': [8, 9, 11],
    'classifier__learning_rate': [0.05, 0.1, 0.15],
    'classifier__scale_pos_weight': [1, 5, 10, scale],
    'classifier__subsample': [0.8, 1.0],
}

pipeline_xgb_gs = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(eval_metric="logloss"))
])

grid_search = GridSearchCV(
    pipeline_xgb_gs,
    param_grid,
    scoring='f1',
    cv=3,
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train)

print("Meilleurs paramètres :", grid_search.best_params_)
print("Meilleur F1 (cv) :", grid_search.best_score_)

Fitting 3 folds for each of 144 candidates, totalling 432 fits


/opt/anaconda3/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning:

[CV] END classifier__learning_rate=0.05, classifier__max_depth=8, classifier__n_estimators=100, classifier__scale_pos_weight=1, classifier__subsample=0.8; total time=  48.7s
[CV] END classifier__learning_rate=0.05, classifier__max_depth=8, classifier__n_estimators=100, classifier__scale_pos_weight=1, classifier__subsample=1.0; total time=  48.1s


/opt/anaconda3/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


[CV] END classifier__learning_rate=0.05, classifier__max_depth=8, classifier__n_estimators=100, classifier__scale_pos_weight=1, classifier__subsample=0.8; total time=  49.9s
[CV] END classifier__learning_rate=0.05, classifier__max_depth=8, classifier__n_estimators=100, classifier__scale_pos_weight=1, classifier__subsample=0.8; total time=  50.3s
[CV] END classifier__learning_rate=0.05, classifier__max_depth=8, classifier__n_estimators=100, classifier__scale_pos_weight=1, classifier__subsample=1.0; total time=  50.1s
[CV] END classifier__learning_rate=0.05, classifier__max_depth=8, classifier__n_estimators=100, classifier__scale_pos_weight=1, classifier__subsample=1.0; total time=  49.4s
[CV] END classifier__learning_rate=0.05, classifier__max_depth=8, classifier__n_estimators=100, classifier__scale_pos_weight=5, classifier__subsample=0.8; total time=  49.9s
[CV] END classifier__learning_rate=0.05, classifier__max_depth=8, classifier__n_estimators=100, classifier__scale_pos_weight=5, cl

...

[CV] END classifier__learning_rate=0.15, classifier__max_depth=11, classifier__n_estimators=300, classifier__scale_pos_weight=1, classifier__subsample=1.0; total time= 3.5min
[CV] END classifier__learning_rate=0.15, classifier__max_depth=11, classifier__n_estimators=300, classifier__scale_pos_weight=1, classifier__subsample=1.0; total time= 3.5min


/opt/anaconda3/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


[CV] END classifier__learning_rate=0.15, classifier__max_depth=11, classifier__n_estimators=300, classifier__scale_pos_weight=1, classifier__subsample=1.0; total time= 3.5min
[CV] END classifier__learning_rate=0.15, classifier__max_depth=11, classifier__n_estimators=300, classifier__scale_pos_weight=1, classifier__subsample=0.8; total time= 3.7min
[CV] END classifier__learning_rate=0.15, classifier__max_depth=11, classifier__n_estimators=300, classifier__scale_pos_weight=1, classifier__subsample=0.8; total time= 3.7min
[CV] END classifier__learning_rate=0.15, classifier__max_depth=11, classifier__n_estimators=300, classifier__scale_pos_weight=1, classifier__subsample=0.8; total time= 3.7min
[CV] END classifier__learning_rate=0.15, classifier__max_depth=11, classifier__n_estimators=300, classifier__scale_pos_weight=5, classifier__subsample=0.8; total time= 3.7min
[CV] END classifier__learning_rate=0.15, classifier__max_depth=11, classifier__n_estimators=300, classifier__scale_pos_weight

/opt/anaconda3/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


[CV] END classifier__learning_rate=0.15, classifier__max_depth=11, classifier__n_estimators=300, classifier__scale_pos_weight=10, classifier__subsample=1.0; total time= 1.9min
[CV] END classifier__learning_rate=0.15, classifier__max_depth=11, classifier__n_estimators=300, classifier__scale_pos_weight=10, classifier__subsample=1.0; total time= 1.9min
[CV] END classifier__learning_rate=0.15, classifier__max_depth=11, classifier__n_estimators=300, classifier__scale_pos_weight=171.75179856115108, classifier__subsample=0.8; total time= 2.0min
[CV] END classifier__learning_rate=0.15, classifier__max_depth=11, classifier__n_estimators=300, classifier__scale_pos_weight=171.75179856115108, classifier__subsample=1.0; total time= 1.9min
[CV] END classifier__learning_rate=0.15, classifier__max_depth=11, classifier__n_estimators=300, classifier__scale_pos_weight=171.75179856115108, classifier__subsample=0.8; total time= 2.0min
[CV] END classifier__learning_rate=0.15, classifier__max_depth=11, class

Meilleurs paramètres : {'classifier__learning_rate': 0.1, 'classifier__max_depth': 9, 'classifier__n_estimators': 100, 'classifier__scale_pos_weight': 10, 'classifier__subsample': 1.0}
Meilleur F1 (cv) : 0.8422878846054976

Meilleurs paramètres : {'classifier__learning_rate': 0.05, 'classifier__max_depth': 9, 'classifier__n_estimators': 100, 'classifier__scale_pos_weight': 1, 'classifier__subsample': 0.8}
Meilleur F1 (cv) : 0.8426820604183353